# Reference analysis notebook

This notebook is provided as reference analysis code for the accepted paper figures. It is not intended to be a standalone reproduction package. Model weights, LoRA adapters, datasets, and intermediate hidden-state files are not included. Local paths under `data/`, `adapters/`, and `outputs/` should be adjusted to the user's environment.

`MAE` denotes mean attention entropy in this notebook, not mean absolute error. The ARC-Challenge input is expected to be a preprocessed instruction/input/output-style JSON file; this notebook does not implement raw ARC dataset preprocessing.


In [ ]:
# Figure: layer-wise mean attention entropy over ARC-Challenge prompts.
# Required inputs: an ARC-Challenge test JSON file, a base model, and optional LoRA adapter paths.
# The notebook computes attention entropy per layer, aggregates mean and standard error, and saves the figure and tables.

from pathlib import Path
import json
import random
import gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
DATA_PATH = Path("data/ARC-Challenge/test.json")
OUTPUT_DIR = Path("outputs/mae_arc_challenge")
MODEL_SPECS = [
    {"label": "LLaMA3", "adapter_path": None},
    {"label": "LLaMA3+Ours", "adapter_path": Path("adapters/ours")},
]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16
MAX_SAMPLES = None
SEED = 42
MAX_LENGTH = 512
MERGE_LORA = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.set_grad_enabled(False)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
def load_records(path, max_samples=None, seed=42):
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    records = []
    for i, ex in enumerate(data):
        instruction = ex.get("instruction", "")
        input_text = ex.get("input", "")
        answer = ex.get("answer", "")
        output_text = ex.get("output", "")
        if input_text.strip():
            prompt = (
                "Below is an instruction that describes a task, paired with an input that provides further context. "
                "Write a response that appropriately completes the request.\n"
                f"### Instruction: {instruction}\n"
                f"### Input: {input_text}\n"
                "### Response:"
            )
        else:
            prompt = (
                "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n"
                f"### Instruction: {instruction}\n"
                "### Response:"
            )
        records.append({"id": i, "prompt": prompt, "instruction": instruction, "answer": answer, "output": output_text})
    if max_samples is not None and len(records) > max_samples:
        records = random.Random(seed).sample(records, max_samples)
    return records

def load_model(base_model_name, adapter_path=None, dtype=torch.float16, device="cuda", merge_lora=True):
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=dtype, output_attentions=True, device_map=None)
    if adapter_path is not None:
        model = PeftModel.from_pretrained(model, str(adapter_path))
        if merge_lora:
            model = model.merge_and_unload()
    model = model.to(device)
    model.eval()
    return model, tokenizer

def mean_attention_entropy(attentions):
    curves = []
    for layer_attn in attentions:
        x = layer_attn[0].detach().float().cpu().numpy()
        head_values = []
        for h in range(x.shape[0]):
            attn = np.clip(x[h], 1e-12, 1.0)
            head_values.append(float(-(attn * np.log(attn)).sum(axis=-1).mean()))
        curves.append(head_values)
    return np.asarray(curves, dtype=np.float64)

def extract_curve(model, tokenizer, prompt, device="cuda", max_length=512):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model(**inputs, output_attentions=True, use_cache=False, return_dict=True)
    layer_head = mean_attention_entropy(outputs.attentions)
    return layer_head.mean(axis=1), int(inputs["input_ids"].shape[1])

def summarize_curve(curve):
    curve = np.asarray(curve, dtype=np.float64)
    mid = curve[9:15]
    return {
        "mae_global_mean": float(curve.mean()),
        "mae_global_std": float(curve.std()),
        "mae_auc": float(curve.sum()),
        "mae_early_mean_L1_3": float(curve[:3].mean()),
        "mae_mid_mean_L10_15": float(mid.mean()),
        "mae_mid_peak_L10_15": float(mid.max()),
        "mae_mid_peak_layer_L10_15": int(mid.argmax() + 10),
        "mae_late_mean_last8": float(curve[-8:].mean()),
        "mae_peak_value": float(curve.max()),
        "mae_peak_layer": int(curve.argmax() + 1),
        "mae_trough_value": float(curve.min()),
        "mae_trough_layer": int(curve.argmin() + 1),
        "mae_drop_L1_to_L3": float(curve[0] - curve[2]),
        "mae_drop_L1_to_midmean": float(curve[0] - mid.mean()),
    }

def analyze_model(records, spec):
    model, tokenizer = load_model(BASE_MODEL_NAME, spec["adapter_path"], DTYPE, DEVICE, MERGE_LORA)
    per_prompt = []
    per_layer = []
    raw_curves = {}
    for n, record in enumerate(records, start=1):
        curve, seq_len = extract_curve(model, tokenizer, record["prompt"], DEVICE, MAX_LENGTH)
        raw_curves[str(record["id"])] = curve.tolist()
        row = summarize_curve(curve)
        row.update({"model": spec["label"], "sample_id": record["id"], "seq_len": seq_len})
        per_prompt.append(row)
        for layer, value in enumerate(curve, start=1):
            per_layer.append({"model": spec["label"], "sample_id": record["id"], "layer": layer, "mae": float(value), "seq_len": seq_len})
        if n % 100 == 0 or n == len(records):
            print(f"{spec['label']}: {n}/{len(records)}")
    del model
    torch.cuda.empty_cache()
    gc.collect()
    layer_df = pd.DataFrame(per_layer)
    summary = layer_df.groupby(["model", "layer"], as_index=False).agg(mae_mean=("mae", "mean"), mae_std=("mae", "std"), n=("mae", "count"))
    summary["mae_se"] = summary["mae_std"].fillna(0) / np.sqrt(summary["n"].clip(lower=1))
    return pd.DataFrame(per_prompt), layer_df, summary, raw_curves

In [ ]:
records = load_records(DATA_PATH, MAX_SAMPLES, SEED)
results = [analyze_model(records, spec) for spec in MODEL_SPECS]
per_prompt_table = pd.concat([r[0] for r in results], ignore_index=True)
per_layer_table = pd.concat([r[1] for r in results], ignore_index=True)
layer_summary_table = pd.concat([r[2] for r in results], ignore_index=True)
raw_curves = {MODEL_SPECS[i]["label"]: results[i][3] for i in range(len(MODEL_SPECS))}

per_prompt_table.to_csv(OUTPUT_DIR / "mae_per_prompt.csv", index=False)
per_layer_table.to_csv(OUTPUT_DIR / "mae_per_layer.csv", index=False)
layer_summary_table.to_csv(OUTPUT_DIR / "mae_layer_summary.csv", index=False)
with (OUTPUT_DIR / "mae_raw_curves.json").open("w", encoding="utf-8") as f:
    json.dump(raw_curves, f, indent=2)

plt.figure(figsize=(9, 5.5))
for spec in MODEL_SPECS:
    label = spec["label"]
    sub = layer_summary_table[layer_summary_table["model"] == label].sort_values("layer")
    x = sub["layer"].to_numpy()
    y = sub["mae_mean"].to_numpy()
    se = sub["mae_se"].fillna(0).to_numpy()
    plt.plot(x, y, marker="o", linewidth=2, label=label)
    plt.fill_between(x, y - se, y + se, alpha=0.2)
plt.xlabel("Layer")
plt.ylabel("Mean Attention Entropy")
plt.title("Layer-wise Mean Attention Entropy")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "layerwise_mae_mean_se.png", dpi=220, bbox_inches="tight")
plt.show()